In [1]:
import pandas as pd

# Read just the first 5 rows of the November file to see its structure
# nrows=5 tells pandas to stop after 5 rows instead of reading the whole 9GB file
df_preview = pd.read_csv("../data/2019-Nov.csv", nrows=5)

df_preview

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-11-01 00:00:00 UTC,view,1003461,2053013555631882655,electronics.smartphone,xiaomi,489.07,520088904,4d3b30da-a5e4-49df-b1a8-ba5943f1dd33
1,2019-11-01 00:00:00 UTC,view,5000088,2053013566100866035,appliances.sewing_machine,janome,293.65,530496790,8e5f4f83-366c-4f70-860e-ca7417414283
2,2019-11-01 00:00:01 UTC,view,17302664,2053013553853497655,NaN,creed,28.31,561587266,755422e7-9040-477b-9bd2-6a6e8fd97387
3,2019-11-01 00:00:01 UTC,view,3601530,2053013563810775923,appliances.kitchen.washer,lg,712.87,518085591,3bfb58cd-7892-48cc-8020-2f17e6de6e7f
4,2019-11-01 00:00:01 UTC,view,1004775,2053013555631882655,electronics.smartphone,xiaomi,183.27,558856683,313628f1-68b8-460d-84f6-cec7a8796ef2


In [2]:
# We'll read the file in chunks of 1 million rows at a time, instead of all at once.
# chunksize tells pandas to return an iterator - something we can loop over -
# rather than one giant DataFrame.

chunk_iter = pd.read_csv("../data/2019-Nov.csv", chunksize=1_000_000)

total_rows = 0
missing_category_code = 0

# Loop through each chunk one at a time
for chunk in chunk_iter:
    total_rows += len(chunk)
    missing_category_code += chunk["category_code"].isna().sum()

print(f"Total rows: {total_rows:,}")
print(f"Missing category_code: {missing_category_code:,}")
print(f"Missing %: {missing_category_code / total_rows * 100:.2f}%")

Total rows: 67,501,979
Missing category_code: 21,898,171
Missing %: 32.44%


In [3]:
# We'll count how many times each event_type appears, across the whole file,
# using the same chunked approach as before.

chunk_iter = pd.read_csv("../data/2019-Nov.csv", chunksize=1_000_000)

# We'll keep a running tally per event type.
# pd.Series() with dtype=int gives us an empty counter we can add to each loop.
event_counts = pd.Series(dtype=int)

for chunk in chunk_iter:
    # value_counts() counts how many times each unique value appears in this chunk
    chunk_counts = chunk["event_type"].value_counts()
    
    # add() combines this chunk's counts with our running total
    # fill_value=0 means "if an event_type hasn't appeared yet, treat it as 0" 
    # rather than throwing an error
    event_counts = event_counts.add(chunk_counts, fill_value=0)

print(event_counts)
print()
print("As percentages:")
print((event_counts / event_counts.sum() * 100).round(2))

event_type
cart         3028930.0
purchase      916939.0
view        63556110.0
dtype: float64

As percentages:
event_type
cart         4.49
purchase     1.36
view        94.15
dtype: float64


In [4]:
# Same event_type count as before, but now on October instead of November
chunk_iter = pd.read_csv(
    "../data/2019-Oct.csv",
    usecols=["event_type"],
    chunksize=1_000_000
)

event_counts_oct = pd.Series(dtype=int)

for chunk in chunk_iter:
    chunk_counts = chunk["event_type"].value_counts()
    event_counts_oct = event_counts_oct.add(chunk_counts, fill_value=0)

print(event_counts_oct)
print()
print("As percentages:")
print((event_counts_oct / event_counts_oct.sum() * 100).round(2))

event_type
cart          926516.0
purchase      742849.0
view        40779399.0
dtype: float64

As percentages:
event_type
cart         2.18
purchase     1.75
view        96.07
dtype: float64


In [5]:
# Check price for garbage values: negatives, zeros, and extreme outliers.
# We'll track running min/max/sum/count and counts of negative/zero prices
# across chunks - these are all "accumulate-safe" stats.
# (Median/percentiles are NOT accumulate-safe this way - more on that below.)

chunk_iter = pd.read_csv(
    "../data/2019-Nov.csv",
    usecols=["price"],
    chunksize=1_000_000
)

price_min = float("inf")
price_max = float("-inf")
price_sum = 0
price_count = 0
negative_count = 0
zero_count = 0

for chunk in chunk_iter:
    prices = chunk["price"]
    
    price_min = min(price_min, prices.min())
    price_max = max(price_max, prices.max())
    price_sum += prices.sum()
    price_count += len(prices)
    negative_count += (prices < 0).sum()
    zero_count += (prices == 0).sum()

print(f"Min price: {price_min}")
print(f"Max price: {price_max}")
print(f"Mean price: {price_sum / price_count:.2f}")
print(f"Negative prices: {negative_count:,}")
print(f"Zero prices: {zero_count:,}")

Min price: 0.0
Max price: 2574.07
Mean price: 292.46
Negative prices: 0
Zero prices: 188,088


In [6]:
chunk_iter = pd.read_csv(
    "../data/2019-Nov.csv",
    usecols=["price", "event_type"],
    chunksize=1_000_000
)

price_min = float("inf")
price_max = float("-inf")
price_sum = 0
price_count = 0
negative_count = 0
zero_count = 0
zero_purchase_count = 0   # <-- new: tracks zero-price PURCHASES specifically

for chunk in chunk_iter:
    prices = chunk["price"]
    
    price_min = min(price_min, prices.min())
    price_max = max(price_max, prices.max())
    price_sum += prices.sum()
    price_count += len(prices)
    negative_count += (prices < 0).sum()
    zero_count += (prices == 0).sum()
    
    # New: filter rows where price is 0 AND event_type is "purchase"
    zero_purchases = chunk[(chunk["price"] == 0) & (chunk["event_type"] == "purchase")]
    zero_purchase_count += len(zero_purchases)

print(f"Min price: {price_min}")
print(f"Max price: {price_max}")
print(f"Mean price: {price_sum / price_count:.2f}")
print(f"Negative prices: {negative_count:,}")
print(f"Zero prices (all events): {zero_count:,}")
print(f"Zero prices (purchases only): {zero_purchase_count:,}")

Min price: 0.0
Max price: 2574.07
Mean price: 292.46
Negative prices: 0
Zero prices (all events): 188,088
Zero prices (purchases only): 0


In [7]:
import duckdb
print(duckdb.__version__)

1.5.5


In [8]:
import duckdb

# DuckDB can read a CSV and write it out as Parquet directly.
# This scans the CSV once, infers types, and writes a compressed columnar file.
# COPY ... TO ... is DuckDB's SQL syntax for exporting query results to a file.

duckdb.sql("""
    COPY (SELECT * FROM read_csv_auto('../data/2019-Nov.csv'))
    TO '../data/2019-Nov.parquet'
    (FORMAT PARQUET)
""")

print("Done converting November")

Done converting November


In [9]:
import os

csv_size = os.path.getsize("../data/2019-Nov.csv")
parquet_size = os.path.getsize("../data/2019-Nov.parquet")

# Convert bytes to GB for readability
csv_gb = csv_size / (1024**3)
parquet_gb = parquet_size / (1024**3)

print(f"CSV size:     {csv_gb:.2f} GB")
print(f"Parquet size: {parquet_gb:.2f} GB")
print(f"Compression ratio: {csv_size / parquet_size:.1f}x smaller")

CSV size:     8.39 GB
Parquet size: 2.29 GB
Compression ratio: 3.7x smaller


In [10]:
duckdb.sql("""
    COPY (SELECT * FROM read_csv_auto('../data/2019-Oct.csv'))
    TO '../data/2019-Oct.parquet'
    (FORMAT PARQUET)
""")

print("Done converting October")

Done converting October


In [11]:
csv_size_oct = os.path.getsize("../data/2019-Oct.csv")
parquet_size_oct = os.path.getsize("../data/2019-Oct.parquet")

csv_gb_oct = csv_size_oct / (1024**3)
parquet_gb_oct = parquet_size_oct / (1024**3)

print(f"CSV size:     {csv_gb_oct:.2f} GB")
print(f"Parquet size: {parquet_gb_oct:.2f} GB")
print(f"Compression ratio: {csv_size_oct / parquet_size_oct:.1f}x smaller")

CSV size:     5.28 GB
Parquet size: 1.42 GB
Compression ratio: 3.7x smaller


In [12]:
result = duckdb.sql("""
    SELECT
        COUNT(*) AS total_rows,
        MIN(price) AS min_price,
        MAX(price) AS max_price,
        AVG(price) AS mean_price,
        MEDIAN(price) AS median_price
    FROM '../data/2019-Nov.parquet'
""")

result.show()

┌────────────┬───────────┬───────────┬───────────────────┬──────────────┐
│ total_rows │ min_price │ max_price │    mean_price     │ median_price │
│   int64    │  double   │  double   │      double       │    double    │
├────────────┼───────────┼───────────┼───────────────────┼──────────────┤
│   67501979 │       0.0 │   2574.07 │ 292.4593165649051 │       165.77 │
└────────────┴───────────┴───────────┴───────────────────┴──────────────┘



In [13]:
result = duckdb.sql("""
    SELECT
        MIN(price) AS min_price,
        QUANTILE_CONT(price, 0.10) AS p10,
        QUANTILE_CONT(price, 0.25) AS p25,
        MEDIAN(price) AS p50_median,
        QUANTILE_CONT(price, 0.75) AS p75,
        QUANTILE_CONT(price, 0.90) AS p90,
        QUANTILE_CONT(price, 0.99) AS p99,
        MAX(price) AS max_price
    FROM '../data/2019-Nov.parquet'
""")

result.show()

┌───────────┬────────┬────────┬────────────┬────────┬────────┬─────────┬───────────┐
│ min_price │  p10   │  p25   │ p50_median │  p75   │  p90   │   p99   │ max_price │
│  double   │ double │ double │   double   │ double │ double │ double  │  double   │
├───────────┼────────┼────────┼────────────┼────────┼────────┼─────────┼───────────┤
│       0.0 │  32.69 │  69.24 │     165.77 │ 360.34 │ 756.75 │ 1661.12 │   2574.07 │
└───────────┴────────┴────────┴────────────┴────────┴────────┴─────────┴───────────┘



In [14]:
# DESCRIBE shows us the schema (column names + types) of a table or file,
# without reading any actual data - just the structure.

result = duckdb.sql("""
    DESCRIBE SELECT * FROM '../data/2019-Nov.parquet'
""")

result.show()

┌───────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name  │ column_type │  null   │   key   │ default │  extra  │
│    varchar    │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ event_time    │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ event_type    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ product_id    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ category_id   │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ category_code │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ brand         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ price         │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ user_id       │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ user_session  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└───────────────┴─────────────┴─────────┴─────────┴─────────┴───

In [15]:
result = duckdb.sql("""
    SELECT
        MAX(product_id) AS max_product_id,
        MAX(category_id) AS max_category_id,
        MAX(user_id) AS max_user_id
    FROM '../data/2019-Nov.parquet'
""")

result.show()

┌────────────────┬─────────────────────┬─────────────┐
│ max_product_id │   max_category_id   │ max_user_id │
│     int64      │        int64        │    int64    │
├────────────────┼─────────────────────┼─────────────┤
│      100028554 │ 2187707861038006932 │   579969851 │
└────────────────┴─────────────────────┴─────────────┘



In [16]:
# Rebuild the November Parquet file with explicit, right-sized types.
# CAST(column AS TYPE) forces a column to a specific type instead of
# trusting auto-detection.
# We're downsizing product_id and user_id from BIGINT to INTEGER
# since we confirmed their max values fit comfortably.
# category_id stays BIGINT - its values are too large to shrink.
# price moves from DOUBLE to FLOAT - we don't need double precision for currency.

duckdb.sql("""
    COPY (
        SELECT
            event_time,
            event_type,
            CAST(product_id AS INTEGER) AS product_id,
            category_id,
            category_code,
            brand,
            CAST(price AS FLOAT) AS price,
            CAST(user_id AS INTEGER) AS user_id,
            user_session
        FROM '../data/2019-Nov.parquet'
    )
    TO '../data/2019-Nov_typed.parquet'
    (FORMAT PARQUET)
""")

print("Done rebuilding November with explicit types")

Done rebuilding November with explicit types


In [17]:
original_size = os.path.getsize("../data/2019-Nov.parquet")
typed_size = os.path.getsize("../data/2019-Nov_typed.parquet")

original_mb = original_size / (1024**2)
typed_mb = typed_size / (1024**2)

print(f"Original Parquet: {original_mb:.1f} MB")
print(f"Typed Parquet:     {typed_mb:.1f} MB")
print(f"Reduction: {(1 - typed_size/original_size) * 100:.1f}%")

Original Parquet: 2346.0 MB
Typed Parquet:     2306.9 MB
Reduction: 1.7%


In [18]:
result = duckdb.sql("""
    SELECT
        (SELECT COUNT(*) FROM '../data/2019-Nov.parquet') AS original_rows,
        (SELECT COUNT(*) FROM '../data/2019-Nov_typed.parquet') AS typed_rows,
        (SELECT SUM(price) FROM '../data/2019-Nov.parquet') AS original_price_sum,
        (SELECT SUM(price) FROM '../data/2019-Nov_typed.parquet') AS typed_price_sum
""")

result.show()

┌───────────────┬────────────┬────────────────────┬───────────────────┐
│ original_rows │ typed_rows │ original_price_sum │  typed_price_sum  │
│     int64     │   int64    │       double       │      double       │
├───────────────┼────────────┼────────────────────┼───────────────────┤
│      67501979 │   67501979 │ 19741582645.121464 │ 19741582638.41387 │
└───────────────┴────────────┴────────────────────┴───────────────────┘



In [19]:
duckdb.sql("""
    COPY (
        SELECT
            event_time,
            event_type,
            CAST(product_id AS INTEGER) AS product_id,
            category_id,
            category_code,
            brand,
            CAST(price AS FLOAT) AS price,
            CAST(user_id AS INTEGER) AS user_id,
            user_session
        FROM '../data/2019-Oct.parquet'
    )
    TO '../data/2019-Oct_typed.parquet'
    (FORMAT PARQUET)
""")

print("Done rebuilding October with explicit types")

Done rebuilding October with explicit types


In [20]:
result = duckdb.sql("""
    SELECT
        (SELECT COUNT(*) FROM '../data/2019-Oct.parquet') AS original_rows,
        (SELECT COUNT(*) FROM '../data/2019-Oct_typed.parquet') AS typed_rows,
        (SELECT SUM(price) FROM '../data/2019-Oct.parquet') AS original_price_sum,
        (SELECT SUM(price) FROM '../data/2019-Oct_typed.parquet') AS typed_price_sum
""")

result.show()

┌───────────────┬────────────┬────────────────────┬───────────────────┐
│ original_rows │ typed_rows │ original_price_sum │  typed_price_sum  │
│     int64     │   int64    │       double       │      double       │
├───────────────┼────────────┼────────────────────┼───────────────────┤
│      42448764 │   42448764 │  12323880556.03752 │ 12323880551.83868 │
└───────────────┴────────────┴────────────────────┴───────────────────┘



In [22]:
result = duckdb.sql("""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN category_id IS NULL THEN 1 ELSE 0 END) AS missing_category_id,
        SUM(CASE WHEN category_code IS NULL THEN 1 ELSE 0 END) AS missing_category_code
    FROM '../data/2019-Nov_typed.parquet'
""")

result.show()

┌────────────┬─────────────────────┬───────────────────────┐
│ total_rows │ missing_category_id │ missing_category_code │
│   int64    │       int128        │        int128         │
├────────────┼─────────────────────┼───────────────────────┤
│   67501979 │                   0 │              21898171 │
└────────────┴─────────────────────┴───────────────────────┘



In [23]:
result = duckdb.sql("""
    SELECT
        category_id,
        COUNT(DISTINCT category_code) AS distinct_codes
    FROM '../data/2019-Nov_typed.parquet'
    WHERE category_code IS NOT NULL
    GROUP BY category_id
    HAVING COUNT(DISTINCT category_code) > 1
""")

result.show()

┌─────────────┬────────────────┐
│ category_id │ distinct_codes │
│    int64    │     int64      │
└─────────────┴────────────────┘
             0 rows           



In [24]:
result = duckdb.sql("""
    WITH category_lookup AS (
        SELECT DISTINCT category_id, category_code
        FROM '../data/2019-Nov_typed.parquet'
        WHERE category_code IS NOT NULL
    )
    SELECT
        COUNT(*) AS total_missing_rows,
        SUM(CASE WHEN lookup.category_code IS NOT NULL THEN 1 ELSE 0 END) AS recoverable,
        SUM(CASE WHEN lookup.category_code IS NULL THEN 1 ELSE 0 END) AS truly_unknown
    FROM '../data/2019-Nov_typed.parquet' AS main
    LEFT JOIN category_lookup AS lookup
        ON main.category_id = lookup.category_id
    WHERE main.category_code IS NULL
""")

result.show()

┌────────────────────┬─────────────┬───────────────┐
│ total_missing_rows │ recoverable │ truly_unknown │
│       int64        │   int128    │    int128     │
├────────────────────┼─────────────┼───────────────┤
│           21898171 │           0 │      21898171 │
└────────────────────┴─────────────┴───────────────┘



In [25]:
# We want to check: how many DISTINCT category_ids are involved in the 
# "missing category_code" rows, and do any of the SAME category_ids 
# ever appear with a non-null category_code anywhere in the dataset?
# If our theory is right, these two groups of category_ids should be 
# completely separate - zero overlap.

result = duckdb.sql("""
    WITH missing_ids AS (
        SELECT DISTINCT category_id
        FROM '../data/2019-Nov_typed.parquet'
        WHERE category_code IS NULL
    ),
    coded_ids AS (
        SELECT DISTINCT category_id
        FROM '../data/2019-Nov_typed.parquet'
        WHERE category_code IS NOT NULL
    )
    SELECT
        (SELECT COUNT(*) FROM missing_ids) AS distinct_ids_with_missing_code,
        (SELECT COUNT(*) FROM coded_ids) AS distinct_ids_with_code,
        (SELECT COUNT(*) FROM missing_ids m 
         INNER JOIN coded_ids c ON m.category_id = c.category_id) AS overlap_ids
""")

result.show()

┌────────────────────────────────┬────────────────────────┬─────────────┐
│ distinct_ids_with_missing_code │ distinct_ids_with_code │ overlap_ids │
│             int64              │         int64          │    int64    │
├────────────────────────────────┼────────────────────────┼─────────────┤
│                            409 │                    275 │           0 │
└────────────────────────────────┴────────────────────────┴─────────────┘



In [26]:
# Combine October and November into a single bronze table.
# UNION ALL stacks the two result sets on top of each other (keeps all rows,
# doesn't remove duplicates - that's different from plain UNION, which dedupes).
# We add a source_file column to each half BEFORE combining, purely for
# traceability - so we can always trace a row back to its origin file later.

duckdb.sql("""
    COPY (
        SELECT *, '2019-Oct' AS source_file FROM '../data/2019-Oct_typed.parquet'
        UNION ALL
        SELECT *, '2019-Nov' AS source_file FROM '../data/2019-Nov_typed.parquet'
    )
    TO '../data/events_bronze.parquet'
    (FORMAT PARQUET)
""")

print("Done combining October and November into events_bronze.parquet")

Done combining October and November into events_bronze.parquet


In [27]:
# Reconciliation: combined row count should exactly equal 
# October's rows + November's rows, with nothing lost or duplicated.

result = duckdb.sql("""
    SELECT
        (SELECT COUNT(*) FROM '../data/2019-Oct_typed.parquet') AS oct_rows,
        (SELECT COUNT(*) FROM '../data/2019-Nov_typed.parquet') AS nov_rows,
        (SELECT COUNT(*) FROM '../data/events_bronze.parquet') AS combined_rows
""")

result.show()

┌──────────┬──────────┬───────────────┐
│ oct_rows │ nov_rows │ combined_rows │
│  int64   │  int64   │     int64     │
├──────────┼──────────┼───────────────┤
│ 42448764 │ 67501979 │     109950743 │
└──────────┴──────────┴───────────────┘



In [28]:
# Apply the category_code fix decided in Decision 1:
# Where category_code is missing, fill it with "unknown_<category_id>"
# instead of leaving it null or using one generic "Unknown" for everything.
# This preserves the fact that there are 409 distinct uncoded categories,
# not one single unknown bucket.
# COALESCE(a, b) returns a if it's not null, otherwise falls back to b.

duckdb.sql("""
    COPY (
        SELECT
            * EXCLUDE (category_code),
            COALESCE(category_code, 'unknown_' || CAST(category_id AS VARCHAR)) AS category_code
        FROM '../data/events_bronze.parquet'
    )
    TO '../data/events_silver.parquet'
    (FORMAT PARQUET)
""")

print("Done creating events_silver.parquet with category_code fix applied")

Done creating events_silver.parquet with category_code fix applied


In [29]:
# Verify the category_code fix worked:
# 1. Confirm zero nulls remain in category_code
# 2. Confirm the total row count is unchanged (no rows lost in the rebuild)
# 3. Count how many rows now carry an "unknown_" label

result = duckdb.sql("""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN category_code IS NULL THEN 1 ELSE 0 END) AS still_null,
        SUM(CASE WHEN category_code LIKE 'unknown_%' THEN 1 ELSE 0 END) AS unknown_labeled
    FROM '../data/events_silver.parquet'
""")

result.show()

┌────────────┬────────────┬─────────────────┐
│ total_rows │ still_null │ unknown_labeled │
│   int64    │   int128   │     int128      │
├────────────┼────────────┼─────────────────┤
│  109950743 │          0 │        35413780 │
└────────────┴────────────┴─────────────────┘



In [30]:
# Spot-check: look at a handful of rows that got the "unknown_" treatment
result = duckdb.sql("""
    SELECT category_id, category_code
    FROM '../data/events_silver.parquet'
    WHERE category_code LIKE 'unknown_%'
    LIMIT 5
""")

result.show()

┌─────────────────────┬─────────────────────────────┐
│     category_id     │        category_code        │
│        int64        │           varchar           │
├─────────────────────┼─────────────────────────────┤
│ 2103807459595387724 │ unknown_2103807459595387724 │
│ 2053013553853497655 │ unknown_2053013553853497655 │
│ 2053013558031024687 │ unknown_2053013558031024687 │
│ 2103807459595387724 │ unknown_2103807459595387724 │
│ 2053013561638126333 │ unknown_2053013561638126333 │
└─────────────────────┴─────────────────────────────┘



In [31]:

# Check for exact duplicate rows: rows that are identical across ALL columns.
# We group by every column, and count how many rows fall into each group.
# If a group has more than 1 row, those rows are exact duplicates of each other.
# GROUP BY ALL is a DuckDB shortcut meaning "group by every column in the SELECT".

result = duckdb.sql("""
    SELECT COUNT(*) AS total_rows_involved_in_duplicates
    FROM (
        SELECT *, COUNT(*) AS dup_count
        FROM '../data/events_silver.parquet'
        GROUP BY ALL
    )
    WHERE dup_count > 1
""")

result.show()

┌───────────────────────────────────┐
│ total_rows_involved_in_duplicates │
│               int64               │
├───────────────────────────────────┤
│                             75652 │
└───────────────────────────────────┘



In [32]:
# 1. Count how many distinct duplicate GROUPS exist (not just total rows involved)
# 2. Look at the actual duplicate rows to see what they look like

result = duckdb.sql("""
    SELECT *, COUNT(*) AS dup_count
    FROM '../data/events_silver.parquet'
    GROUP BY ALL
    HAVING COUNT(*) > 1
    ORDER BY dup_count DESC
    LIMIT 10
""")

result.show()

┌─────────────────────┬────────────┬────────────┬─────────────────────┬─────────┬────────┬───────────┬──────────────────────────────────────┬─────────────┬───────────────────────────────┬───────────┐
│     event_time      │ event_type │ product_id │     category_id     │  brand  │ price  │  user_id  │             user_session             │ source_file │         category_code         │ dup_count │
│      timestamp      │  varchar   │   int32    │        int64        │ varchar │ float  │   int32   │               varchar                │   varchar   │            varchar            │   int64   │
├─────────────────────┼────────────┼────────────┼─────────────────────┼─────────┼────────┼───────────┼──────────────────────────────────────┼─────────────┼───────────────────────────────┼───────────┤
│ 2019-11-17 16:58:15 │ cart       │    1004836 │ 2053013555631882655 │ samsung │ 244.02 │ 518642416 │ 77d81adc-098e-4ae7-b608-2b925aa236f2 │ 2019-Nov    │ electronics.smartphone        │        78 │


In [33]:
# Check whether duplicate rows are concentrated in specific event_types,
# or spread evenly across view/cart/purchase.
# We reuse the same duplicate-detection logic, but now group the results
# by event_type to see where duplication is actually happening.

result = duckdb.sql("""
    SELECT
        event_type,
        COUNT(*) AS duplicate_rows
    FROM (
        SELECT *, COUNT(*) AS dup_count
        FROM '../data/events_silver.parquet'
        GROUP BY ALL
    )
    WHERE dup_count > 1
    GROUP BY event_type
    ORDER BY duplicate_rows DESC
""")

result.show()

┌────────────┬────────────────┐
│ event_type │ duplicate_rows │
│  varchar   │     int64      │
├────────────┼────────────────┤
│ cart       │          71961 │
│ view       │           3606 │
│ purchase   │             85 │
└────────────┴────────────────┘

